# Experiment configuration

In [ ]:
from pathlib import Path
import gc
import json
import random
import shutil
import time
import torch_geometric
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import FileLink, display
from torch import nn
from torch_geometric.nn import GCNConv, GraphConv, SAGEConv

# Experiment settings
RUN_DATES = ["2022-06-17", "2023-08-15", "2024-08-25"]
DATES = ["2022-06-17", "2023-08-15", "2024-08-25"]
GROUPS = ["all", "public"]
MODELS = ["MLP", "GraphSAGE", "GCN", "GraphConv"]
SPLITS = [42, 123, 2026, 3407, 777]
INIT_SEEDS = [11, 22, 33]

N_TRIALS = 20
TUNE_MAX_EPOCHS = 1500
FINAL_MAX_EPOCHS = 3000
TUNING_INIT_SEED = 11

DROPOUT = 0.1
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0
LOG_EVERY = 500

# find data
candidates = sorted({
    p.parent for p in Path("/kaggle/input").rglob("experiment_config.json")
    if p.parent.name == "static_grid90_three_dates_v1"
})
if not candidates:
    raise FileNotFoundError("no data")
DATA = candidates[0]

OUT = Path("/kaggle/working/results_grid90_three_dates_v1")
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__ if "torch_geometric" in globals() else "installed")
print("Optuna:", optuna.__version__)
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("Data dir:", DATA)

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save experiment config and keep a copy of the input data
CONFIG = {
    "version": "grid90_three_dates_training_v1",
    "dates": DATES, "feature_groups": GROUPS, "models": MODELS,
    "split_seeds": SPLITS, "init_seeds": INIT_SEEDS,
    "n_trials": N_TRIALS, "tuning_init_seed": TUNING_INIT_SEED,
    "tuning_max_epochs": TUNE_MAX_EPOCHS, "final_max_epochs": FINAL_MAX_EPOCHS,
    "dropout": DROPOUT, "weight_decay": WEIGHT_DECAY, "gradient_clip": GRAD_CLIP,
    "optimizer": "AdamW", "graph": "grid8_unit_weights",
}
write_json(OUT / "experiment_config.json", CONFIG)
shutil.copytree(DATA, OUT / "data_snapshot", dirs_exist_ok=True)

## Load graph, data loader, and model definition

In [ ]:
# Load common node order and 8-neighbour grid graph
nodes = pd.read_csv(DATA / "node_order.csv", dtype={"grid_id": str})
nodes = nodes.sort_values("local_node_id").reset_index(drop=True)
N = len(nodes)

ei = np.load(DATA / "graphs" / "edge_index_grid8.npy")
ew = np.load(DATA / "graphs" / "edge_weight_grid8.npy")
EDGE = torch.as_tensor(ei, dtype=torch.long, device=DEVICE)
WEIGHT = torch.as_tensor(ew, dtype=torch.float32, device=DEVICE)

def load_data(date, group, split):
    path = DATA / date / group / f"split_{split}.npz"
    with np.load(path, allow_pickle=False) as z:
        a = {k: z[k].copy() for k in z.files}

    features = read_json(DATA / date / group / "feature_names.json")
    masks = {
        "train": a["mask_train"].astype(bool),
        "val": a["mask_valid"].astype(bool),
        "test": a["mask_test"].astype(bool),
    }
    assert np.all(masks["train"].astype(int) + masks["val"].astype(int) + masks["test"].astype(int) == 1)

    train = masks["train"]
    ym = float(a["y_raw"][train].mean())
    ys = float(a["y_raw"][train].std(ddof=0))
    if ys < 1e-8:
        ys = 1.0

    return {
        "X": torch.as_tensor(a["X"], dtype=torch.float32, device=DEVICE),
        "y": torch.as_tensor(a["y"], dtype=torch.float32, device=DEVICE),
        "masks": {k: torch.as_tensor(m, dtype=torch.bool, device=DEVICE) for k, m in masks.items()},
        "masks_np": masks,
        "y_raw": a["y_raw"],
        "y_mean": ym,
        "y_std": ys,
        "features": features,
    }
###############################
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
#########################################################################
class Regressor(nn.Module):
    def __init__(self, name, input_dim, params):
        super().__init__()
        self.name = name
        self.activation = params["activation"]
        hidden = int(params["hidden_dim"])
        self.layers = nn.ModuleList()

        for i in range(int(params["n_layers"])):
            in_dim = input_dim if i == 0 else hidden
            if name == "MLP":
                layer = nn.Linear(in_dim, hidden)
            elif name == "GraphSAGE":
                layer = SAGEConv(in_dim, hidden, aggr="mean")
            elif name == "GCN":
                layer = GCNConv(in_dim, hidden, cached=True, add_self_loops=True, normalize=True)
            elif name == "GraphConv":
                layer = GraphConv(in_dim, hidden, aggr="add")
            else:
                raise ValueError(f"Unknown model: {name}")
            self.layers.append(layer)

        self.output = nn.Linear(hidden, 1)

    def forward(self, x):
        for layer in self.layers:
            if self.name == "MLP":
                x = layer(x)
            elif self.name == "GraphSAGE":
                x = layer(x, EDGE)
            else:
                x = layer(x, EDGE, edge_weight=WEIGHT)

            if self.activation == "relu":
                x = F.relu(x)
            elif self.activation == "elu":
                x = F.elu(x)
            else:
                x = F.gelu(x)

            x = F.dropout(x, p=DROPOUT, training=self.training)
        return self.output(x).squeeze(-1)

## Single-run training and Optuna tuning

In [ ]:
def fit_one(date, group, name, split, seed, params, data, max_epochs, folder, save_weights=True):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    set_seed(seed)

    model = Regressor(name, data["X"].shape[1], params).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(params["learning_rate"]), weight_decay=WEIGHT_DECAY)
    tr, va = data["masks"]["train"], data["masks"]["val"]

    best_val = float("inf")
    best_epoch = 0
    best_state = None
    stale = 0
    history = []
    stop_reason = "epoch_limit"
    start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        pred = model(data["X"])
        loss = F.mse_loss(pred[tr], data["y"][tr])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = model(data["X"])
            train_mse = F.mse_loss(pred[tr], data["y"][tr]).item()
            val_mse = F.mse_loss(pred[va], data["y"][va]).item()

        val_rmse = float(np.sqrt(val_mse) * data["y_std"])
        history.append({
            "epoch": epoch,
            "train_MSE_standardized": float(loss.item()),
            "train_RMSE_C": float(np.sqrt(train_mse) * data["y_std"]),
            "val_RMSE_C": val_rmse,
        })

        if val_rmse < best_val:
            best_val = val_rmse
            best_epoch = epoch
            stale = 0
            if save_weights:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1

        if epoch % LOG_EVERY == 0:
            print(f"    epoch={epoch} | val={val_rmse:.4f} | best={best_val:.4f}", flush=True)

        if stale >= int(params["patience"]):
            stop_reason = "early_stopping"
            break

    elapsed = time.perf_counter() - start
    pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
    if save_weights:
        torch.save(best_state, folder / "best_model.pt")

    info = {
        "date": date, "feature_group": group, "model": name,
        "graph": "none" if name == "MLP" else "grid8",
        "split_seed": int(split), "init_seed": int(seed),
        "params": params, "feature_names": data["features"],
        "best_epoch": int(best_epoch), "final_epoch": int(epoch),
        "max_epochs": int(max_epochs), "best_val_RMSE_C": float(best_val),
        "stop_reason": stop_reason, "hit_epoch_limit": bool(epoch == max_epochs),
        "seconds": float(elapsed),
        "parameters": int(sum(p.numel() for p in model.parameters())),
        "completed": True,
    }
    write_json(folder / "run.json", info)

    del model, optimizer, pred, loss, best_state
    clear_memory()
    return info

##################################################################
def tune_one(date, group, name, split, data):
    root = OUT / date / group / name / f"split_{split}"
    root.mkdir(parents=True, exist_ok=True)
    params_file = root / "best_params.json"

    # Reuse best params if tuning already finished
    if params_file.exists():
        return read_json(params_file)["params"]

    sampler_seed = split + 10000 * MODELS.index(name)
    study = optuna.create_study(
        study_name="validation_rmse",
        storage=f"sqlite:///{(root / 'optuna.db').as_posix()}",
        direction="minimize",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=sampler_seed, n_startup_trials=5),
        pruner=optuna.pruners.NopPruner(),
    )

    def objective(trial):
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
            "n_layers": trial.suggest_categorical("n_layers", [1, 2, 3]),
            "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
            "activation": trial.suggest_categorical("activation", ["relu", "elu", "gelu"]),
            "patience": trial.suggest_categorical("patience", [50, 100, 150]),
        }
        info = fit_one(
            date, group, name, split, TUNING_INIT_SEED,
            params, data, TUNE_MAX_EPOCHS,
            root / "tuning" / f"trial_{trial.number:04d}",
            save_weights=False,
        )
        return info["best_val_RMSE_C"]

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < N_TRIALS:
        study.optimize(objective, n_trials=N_TRIALS - done, n_jobs=1, gc_after_trial=True)

    study.trials_dataframe().to_csv(root / "optuna_trials.csv", index=False)
    saved = {
        "date": date, "feature_group": group, "model": name, "split_seed": int(split),
        "completed_trials": sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials),
        "best_trial": int(study.best_trial.number),
        "tuning_val_RMSE_C": float(study.best_value),
        "params": study.best_params,
    }
    write_json(params_file, saved)
    return saved["params"]

## Evaluation, aggregation, and backup functions

In [ ]:
def metric_values(y, p):
    err = p - y
    sst = np.sum((y - y.mean()) ** 2)
    return {
        "n": len(y),
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "R2": float(1 - np.sum(err ** 2) / sst) if sst > 0 else np.nan,
        "Bias": float(np.mean(err)),
    }

def evaluate_date(date):
    rows, runs, params_rows = [], [], []

    for group in GROUPS:
        for name in MODELS:
            for split in SPLITS:
                data = load_data(date, group, split)
                root = OUT / date / group / name / f"split_{split}"
                chosen = read_json(root / "best_params.json")
                params = chosen["params"]

                params_rows.append({
                    "date": date, "feature_group": group, "model": name,
                    "split_seed": split, "tuning_val_RMSE_C": chosen["tuning_val_RMSE_C"],
                    **params,
                })

                for seed in INIT_SEEDS:
                    folder = root / f"seed_{seed}"
                    info = read_json(folder / "run.json")

                    model = Regressor(name, data["X"].shape[1], params).to(DEVICE)
                    state = torch.load(folder / "best_model.pt", map_location=DEVICE, weights_only=True)
                    model.load_state_dict(state)
                    model.eval()

                    with torch.no_grad():
                        pred = model(data["X"]).cpu().numpy().astype(float)
                    pred = pred * data["y_std"] + data["y_mean"]

                    base = {
                        "date": date, "feature_group": group, "model": name,
                        "graph": info["graph"], "split_seed": split, "init_seed": seed,
                    }
                    local_rows = []
                    subset_arr = np.full(N, "", dtype="<U5")

                    for subset, mask in data["masks_np"].items():
                        row = {**base, "subset": subset, **metric_values(data["y_raw"][mask], pred[mask])}
                        rows.append(row)
                        local_rows.append(row)
                        subset_arr[mask] = subset

                    pd.DataFrame(local_rows).to_csv(folder / "metrics.csv", index=False)

                    pred_df = nodes.copy()
                    pred_df["date"] = date
                    pred_df["subset"] = subset_arr
                    pred_df["observed_lst"] = data["y_raw"]
                    pred_df["predicted_lst"] = pred
                    pred_df["error_C"] = pred - data["y_raw"]
                    pred_df.to_csv(folder / "predictions.csv", index=False)

                    runs.append({k: v for k, v in info.items() if k not in ["params", "feature_names"]})
                    del model, state, pred

                del data
                clear_memory()

    # Aggregate over init seeds first, then mean and std across splits
    metrics = pd.DataFrame(rows)
    run_table = pd.DataFrame(runs)
    measures = ["MAE", "RMSE", "R2", "Bias"]
    group_keys = ["date", "feature_group", "model", "graph", "subset"]
    split_keys = group_keys + ["split_seed"]

    per_split = metrics.groupby(split_keys, as_index=False)[measures].mean()
    summary = per_split.groupby(group_keys)[measures].agg(["mean", "std"])
    summary.columns = ["_".join(c) for c in summary.columns]
    summary = summary.reset_index()

    # Paired RMSE comparison: public vs all features
    all_part = per_split[per_split["feature_group"] == "all"]
    pub_part = per_split[per_split["feature_group"] == "public"]
    pair_keys = ["date", "model", "graph", "subset", "split_seed"]
    paired = all_part[pair_keys + ["RMSE"]].merge(
        pub_part[pair_keys + ["RMSE"]], on=pair_keys, suffixes=("_all", "_public")
    )
    paired["delta_public_minus_all_C"] = paired["RMSE_public"] - paired["RMSE_all"]

    # Save date-level tables
    date_root = OUT / date
    metrics.to_csv(date_root / "metrics_all_runs.csv", index=False)
    per_split.to_csv(date_root / "metrics_per_split.csv", index=False)
    summary.to_csv(date_root / "summary.csv", index=False)
    run_table.to_csv(date_root / "training_runs.csv", index=False)
    pd.DataFrame(params_rows).to_csv(date_root / "best_params_all_splits.csv", index=False)
    paired.to_csv(date_root / "paired_public_vs_all.csv", index=False)

    write_json(date_root / "evaluation_complete.json", {
        "date": date, "final_runs": len(run_table),
        "initializations_per_split": len(INIT_SEEDS), "node_splits": len(SPLITS),
    })

    print(f"=== Test summary for {date} ===")
    display(summary[summary["subset"] == "test"].sort_values(["feature_group", "RMSE_mean"]).round(4))

def refresh_combined_summary():
    done_dates = [d for d in DATES if (OUT / d / "evaluation_complete.json").exists()]
    filenames = [
        "metrics_all_runs.csv", "metrics_per_split.csv", "summary.csv",
        "training_runs.csv", "best_params_all_splits.csv", "paired_public_vs_all.csv",
    ]
    for fname in filenames:
        tables = [pd.read_csv(OUT / d / fname) for d in done_dates]
        if tables:
            pd.concat(tables, ignore_index=True).to_csv(OUT / fname, index=False)

def backup_results():
    archive = shutil.make_archive(str(OUT) + "_backup", "zip", root_dir=OUT.parent, base_dir=OUT.name)
    display(FileLink(archive))

## Run all selected dates

In [ ]:
for run_date in RUN_DATES:
    print(f"\n========== Running {run_date} ==========")
    for group in GROUPS:
        for name in MODELS:
            for split in SPLITS:
                data = load_data(run_date, group, split)
                params = tune_one(run_date, group, name, split, data)
                root = OUT / run_date / group / name / f"split_{split}"

                for seed in INIT_SEEDS:
                    folder = root / f"seed_{seed}"
                    if (folder / "run.json").exists() and (folder / "best_model.pt").exists():
                        continue
                    fit_one(
                        run_date, group, name, split, seed,
                        params, data, FINAL_MAX_EPOCHS, folder,
                        save_weights=True,
                    )

                del data
                clear_memory()

    evaluate_date(run_date)
    refresh_combined_summary()
    backup_results()